# getting medical simplifcation data from github 

In [34]:
import requests
import os
import datetime
import random
import zipfile
import shutil
import math
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW # Changed from transformers import AdamW
import datasets as hf_datasets # Alias to avoid conflict with local variables
import transformers as hf_transformers # Alias
from datasets import Dataset, DatasetDict, load_dataset
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, set_seed, get_scheduler
from accelerate import Accelerator, notebook_launcher
from tqdm.notebook import tqdm # Or from tqdm import tqdm
import importlib.metadata # For version checking


In [ ]:

user = "SebaJoe"
repo = "MultiCochrane"
branch = "main" 
path_to_dir = "data/MultiCochrane"

filenames = [
    "multiCochrane_all.zip"
]

local_dir = "./data"

try:
    os.makedirs(local_dir, exist_ok=True)
    print(f"Directory '{local_dir}' ensured.")
except OSError as e:
    print(f"Error creating directory {local_dir}: {e}")
    filenames = [] # Prevent download attempts if dir fails

download_count = 0
error_count = 0
for filename in filenames:
    # Construct the URL to the raw file content
    raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/{path_to_dir}/{filename}"

    # Construct the full local path
    local_filepath = os.path.join(local_dir, filename)

    print(f"  Downloading '{filename}' from {raw_url}...")

    try:
        response = requests.get(raw_url, stream=True) 
        response.raise_for_status() 

        # Write the content to the local file
        with open(local_filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"  Successfully saved to '{local_filepath}'")
        download_count += 1

    except requests.exceptions.RequestException as e:
        print(f"  Error downloading {filename}: {e}")
        error_count += 1
    except IOError as e:
         print(f"  Error writing file {local_filepath}: {e}")
         error_count += 1
    except Exception as e:
        print(f"  An unexpected error occurred for {filename}: {e}")
        error_count += 1


print("\n--- Download Summary ---")
print(f"Successfully downloaded: {download_count} file(s)")
print(f"Errors encountered: {error_count} file(s)")
if error_count == 0 and download_count == len(filenames):
    print("All expected files downloaded successfully.")
elif download_count > 0:
     print("Some files downloaded. Check logs for errors.")
else:
     print("No files were downloaded.")

Directory './data' ensured.
  Successfully saved to './data\multiCochrane_all.zip'

--- Download Summary ---
Successfully downloaded: 1 file(s)
Errors encountered: 0 file(s)
All expected files downloaded successfully.


# Loading the data

In [ ]:
#for the new data (muilticochrane_all.zip)
def unzip_file(zip_file_path, extract_to_dir):
    """
    Unzips a specified zip file to a target directory.

    Args:
        zip_file_path (str): The path to the .zip file to be extracted.
        extract_to_dir (str): The directory where the contents should be extracted.
                               If it doesn't exist, it will be created.
    """
    try:
        os.makedirs(extract_to_dir, exist_ok=True)
        print(f"Ensured extraction directory '{extract_to_dir}' exists.")
    except OSError as e:
        print(f"Error creating directory {extract_to_dir}: {e}")
        return # Stop if directory creation fails

    if not os.path.isfile(zip_file_path):
        print(f"Error: Zip file not found at '{zip_file_path}'")
        return

    print(f"Attempting to extract '{zip_file_path}' to '{extract_to_dir}'...")
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to_dir)
        print(f"Successfully extracted '{zip_file_path}' to '{extract_to_dir}'")

    except zipfile.BadZipFile:
        print(f"Error: Failed to unzip. '{zip_file_path}' might be corrupted or not a valid zip file.")
    except FileNotFoundError:
        print(f"Error: Zip file not found at '{zip_file_path}' during opening.")
    except Exception as e:
        print(f"An unexpected error occurred during extraction: {e}")


path_to_my_zip = local_dir + "/" + filenames[0] 
destination_folder = local_dir + "/multiCochrane_all"
unzip_file(path_to_my_zip, destination_folder)

Ensured extraction directory './data/multiCochrane_all' exists.
Attempting to extract './data/multiCochrane_all.zip' to './data/multiCochrane_all'...
Successfully extracted './data/multiCochrane_all.zip' to './data/multiCochrane_all'


In [36]:
base_path = "./data/multiCochrane_all/filtered (r=0.5)/en"
data_files = {
    "train": os.path.join(base_path, "train0.5_en.csv"),
    "test": os.path.join(base_path, "test0.5_en.csv"),
    "validation": os.path.join(base_path, "val0.5_en.csv") 
}

try:
    multi_cochrane_dataset = load_dataset("csv", data_files=data_files)
    print(multi_cochrane_dataset)
except Exception as e:
    print(f"\nAn error occurred during dataset loading: {e}")


DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 30032
    })
    test: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 395
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'prefix', 'input_text', 'target_text', 'doi'],
        num_rows: 83
    })
})


In [42]:
multi_cochrane_dataset["train"][6]

{'Unnamed: 0': 15,
 'prefix': None,
 'input_text': 'We could not say whether more people who had a remote check-up needed oral corticosteroids for an asthma exacerbation than those who were seen face-to-face because the confidence intervals (CIs) were very wide (OR 1.74, 95% CI 0.41 to 7.44; 278 participants; one study; low quality evidence).',
 'target_text': 'We cannot say whether or not people who had a check-up over the phone or internet were more or less likely to need oral corticosteroids for an asthma attack than those seen face-to-face, and we were uncertain of the result for several reasons.',
 'doi': '10.1002/14651858.CD011715.pub2'}

In [ ]:
#for the old data (normal.txt and simple.txt)
import os
import random
import math
import datetime
from datasets import Dataset, DatasetDict
import pandas as pd 

normal_corpus_file = os.path.join(local_dir, "normal.txt")
simple_corpus_file = os.path.join(local_dir, "simple.txt")
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 1.0 - train_ratio - val_ratio # Implicitly 0.1
random_seed = 42
random.seed(random_seed)

print(f"Script running at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Loading data from: {normal_corpus_file} and {simple_corpus_file}")
print(f"Splitting data: Train={train_ratio*100:.1f}%, Val={val_ratio*100:.1f}%, Test={test_ratio*100:.1f}%")
parallel_pairs = []
try:
    with open(normal_corpus_file, 'r', encoding='utf-8') as f_normal, \
         open(simple_corpus_file, 'r', encoding='utf-8') as f_simple:

        normal_lines = f_normal.readlines()
        simple_lines = f_simple.readlines()

        if len(normal_lines) != len(simple_lines):
            print(f"Error: Mismatched line counts! Normal: {len(normal_lines)}, Simple: {len(simple_lines)}")
            exit()

        print(f"Found {len(normal_lines)} lines in each file.")
        processed_count = 0
        skipped_count = 0

        for i in range(len(normal_lines)):
            norm_line_full = normal_lines[i].strip()
            simp_line_full = simple_lines[i].strip()

            norm_parts = norm_line_full.split(None, 2) 
            simp_parts = simp_line_full.split(None, 2) 

            if len(norm_parts) == 3 and len(simp_parts) == 3:
                normal_sentence = norm_parts[2].strip() 
                simple_sentence = simp_parts[2].strip() 
                if normal_sentence and simple_sentence:
                    # Append as a dictionary
                    parallel_pairs.append({'normal': normal_sentence, 'simple': simple_sentence})
                    processed_count += 1
                else:
                    skipped_count += 1 # Count if extracted sentence is empty
            else:
                skipped_count += 1

    print(f"Successfully processed and paired {processed_count} lines.")
    if skipped_count > 0:
        print(f"Skipped {skipped_count} lines due to formatting issues or empty sentences.")


    #ashuffle the pairs
    if not parallel_pairs:
        print("Warning: No data pairs were loaded. Cannot proceed with splitting.")
    else:
        print(f"Shuffling pairs with random seed: {random_seed}...")
        random.shuffle(parallel_pairs)

    #split sizes
    total_pairs = len(parallel_pairs)
    train_end_idx = math.floor(train_ratio * total_pairs)
    val_end_idx = train_end_idx + math.floor(val_ratio * total_pairs)

    #carry oyt splitting
    train_list = parallel_pairs[0:train_end_idx]
    val_list = parallel_pairs[train_end_idx:val_end_idx]
    test_list = parallel_pairs[val_end_idx:]

    print("\n--- Dataset Split Summary (before HF conversion) ---")
    print(f"Training set size:   {len(train_list)} pairs")
    print(f"Validation set size: {len(val_list)} pairs")
    print(f"Test set size:       {len(test_list)} pairs")

    #dataset format
    train_dataset = Dataset.from_list(train_list)
    val_dataset = Dataset.from_list(val_list)
    test_dataset = Dataset.from_list(test_list)

    #dict
    medical_simplification_dataset = DatasetDict({
        'train': train_dataset,
        'validation': val_dataset,
        'test': test_dataset
    })

    #first example in the training set
    print(medical_simplification_dataset)
    if len(medical_simplification_dataset['train']) > 0:
        print("\nExample Training Item from Dataset:")
        print(medical_simplification_dataset['train'][0])
    else:
        print("\nTraining dataset is empty.")


except FileNotFoundError:
    print(f"Error: Could not find '{normal_corpus_file}' or '{simple_corpus_file}'.")
    print("Please ensure the file names are correct and the download script ran successfully.")
except ImportError:
    print("Error: 'datasets' library not found. Please install it using: pip install datasets")
except Exception as e:
    print(f"An error occurred during loading or splitting: {e}")

Script running at: 2025-03-31 21:51:15
Loading data from: ./data\normal.txt and ./data\simple.txt
Splitting data: Train=80.0%, Val=10.0%, Test=10.0%
Found 890 lines in each file.
Successfully processed and paired 890 lines.
Shuffling pairs with random seed: 42...

--- Dataset Split Summary (before HF conversion) ---
Training set size:   712 pairs
Validation set size: 89 pairs
Test set size:       89 pairs
DatasetDict({
    train: Dataset({
        features: ['normal', 'simple'],
        num_rows: 712
    })
    validation: Dataset({
        features: ['normal', 'simple'],
        num_rows: 89
    })
    test: Dataset({
        features: ['normal', 'simple'],
        num_rows: 89
    })
})

Example Training Item from Dataset:
{'normal': '-LRB- Quick Scatterometer -RRB- , an earth-observing satellite , provided estimates of wind speed and direction over the oceans to National Oceanic and Atmospheric Administration and others .', 'simple': '-LRB- Quick Scatterometer -RRB- is an earth-obse

# Encode dataset


In [4]:
#based on https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/T5/Fine_tuning_Dutch_T5_base_on_CNN_Daily_Mail_for_summarization_(on_TPU_using_HuggingFace_Accelerate).ipynb#scrollTo=tiLdcTmkg-_o
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-base")

prefix = "Simplify: "
max_input_length = 512
max_target_length = 512

def preprocess_examples(examples):
  # encode the documents
  normals = examples['normal']
  simplied = examples['simple']
  
  inputs = [prefix + normal for normal in normals]
  model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)

  # encode the simplifications
  labels = tokenizer(simplied, max_length=max_target_length, padding="max_length", truncation=True).input_ids

  # important: we need to replace the index of the padding tokens by -100
  # such that they are not taken into account by the CrossEntropyLoss
  labels_with_ignore_index = []
  for labels_example in labels:
    labels_example = [label if label != 0 else -100 for label in labels_example]
    labels_with_ignore_index.append(labels_example)
  
  model_inputs["labels"] = labels_with_ignore_index

  return model_inputs

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [5]:
train_ds = medical_simplification_dataset['train']
val_ds = medical_simplification_dataset['validation']
test_ds = medical_simplification_dataset['test']
encoded_train_ds = train_ds.map(preprocess_examples, batched=True, remove_columns=train_ds.column_names)
encoded_val_ds = val_ds.map(preprocess_examples, batched=True, remove_columns=val_ds.column_names)
encoded_test_ds = test_ds.map(preprocess_examples, batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

In [6]:
#set format to PyTorch
encoded_train_ds.set_format(type="torch")
encoded_val_ds.set_format(type="torch")
encoded_test_ds.set_format(type="torch")

In [7]:
from torch.utils.data import DataLoader

def create_dataloaders(train_batch_size=8, eval_batch_size=32):
    train_dataloader = DataLoader(encoded_train_ds, shuffle=True, batch_size=train_batch_size)
    val_dataloader = DataLoader(encoded_val_ds, shuffle=False, batch_size=eval_batch_size)
    return train_dataloader, val_dataloader

# FineTuning model

In [8]:
hyperparameters = {
    "learning_rate": 1e-4,
    "num_epochs": 5, 
    "train_batch_size": 16, 
    "gradient_accumulation_steps": 4,
    "eval_batch_size": 32, 
    "seed": 42,
    "patience": 3, 
    "output_dir": "/content_simplifier_t5_base/",
    "lr_scheduler_type": "linear",
    "num_warmup_steps": 500
}

In [11]:
# ==============================================================================
# IMPORTS needed specifically for this block (ensure these are run)
# ==============================================================================
import torch
from torch.optim import AdamW # Using PyTorch's AdamW
import datasets as hf_datasets
import transformers as hf_transformers
from transformers import T5ForConditionalGeneration, set_seed
from accelerate import Accelerator
from tqdm.notebook import tqdm
import os

# ==============================================================================
# 5. Hyperparameters (Corrected - removed unused num_workers)
# ==============================================================================
hyperparameters = {
    "model_checkpoint": "google-t5/t5-base", # Using t5-base now
    "learning_rate": 5e-5, # Often good starting point for base models
    "num_epochs": 3, # Might need fewer epochs for larger models/datasets
    "train_batch_size": 16, # ADJUST based on VRAM for t5-base! Start lower maybe?
    "gradient_accumulation_steps": 1, # Keeping simple loop (1 = no accumulation)
    "eval_batch_size": 32, # ADJUST based on VRAM for t5-base!
    "seed": 42,
    "patience": 3,
    "output_dir": "./content_simplifier_t5_base_simple/", # Updated output dir name
    "mixed_precision": "fp16", # Should enable fp16 on RTX 4090
    # "num_dataloader_workers": 4, # REMOVED - Not used by create_dataloaders in Block 1
}
print("\n--- Hyperparameters ---")
for key, value in hyperparameters.items(): print(f"{key}: {value}")
print("-----------------------")

# ==============================================================================
# 6. Simplified Training Function (Consistent with Block 1 Data Prep)
# ==============================================================================
# Assumes 'tokenizer', 'create_dataloaders', 'encoded_train_ds', 'encoded_val_ds'
# are defined globally based on your first code block.

def training_function():
    global hyperparameters, tokenizer # Access global hyperparameters and tokenizer

    # --- Add Debug Check ---
    mp_value_to_pass = hyperparameters.get("mixed_precision", None)
    # Use print directly here as accelerator might not be fully initialized for accelerator.print
    print(f"[DEBUG] Value being passed to Accelerator for mixed_precision: {mp_value_to_pass}")
    # --- End Debug Check ---

    # Initialize accelerator
    accelerator = Accelerator(
        mixed_precision=mp_value_to_pass
    )

    # --- Accelerator/Device Diagnostics ---
    accelerator.print("\n--- Accelerator Info ---")
    accelerator.print(f"Accelerator device: {accelerator.device}")
    accelerator.print(f"Number of processes: {accelerator.num_processes}")
    accelerator.print(f"Using mixed precision: {accelerator.state.mixed_precision}") # Corrected attribute
    accelerator.print("------------------------")
    # --- End Diagnostics ---

    # Logging verbosity
    if accelerator.is_main_process:
        hf_datasets.utils.logging.set_verbosity_warning()
        hf_transformers.utils.logging.set_verbosity_info()
    else:
        hf_datasets.utils.logging.set_verbosity_error()
        hf_transformers.utils.logging.set_verbosity_error()

    set_seed(hyperparameters["seed"])
    accelerator.print(f"Using seed: {hyperparameters['seed']}")

    # Instantiate model
    model_name = hyperparameters["model_checkpoint"]
    accelerator.print(f"Loading model: {model_name}")
    model = T5ForConditionalGeneration.from_pretrained(model_name)

    # Use the globally loaded tokenizer for saving consistency
    # Make sure 'tokenizer' was loaded successfully in the previous block
    if 'tokenizer' not in globals():
        raise NameError("Global variable 'tokenizer' not found. Ensure it was loaded before calling training_function.")
    active_tokenizer = tokenizer

    # Optimizer - Using torch.optim.AdamW
    optimizer = torch.optim.AdamW(model.parameters(), lr=hyperparameters["learning_rate"])

    # Create DataLoaders using the function defined in Block 1
    # This call matches the function definition in Block 1 (no num_workers)
    # Assumes 'create_dataloaders' function is defined globally
    if 'create_dataloaders' not in globals():
         raise NameError("Function 'create_dataloaders' not found. Ensure it was defined before calling training_function.")
    train_dataloader, val_dataloader = create_dataloaders(
        train_batch_size=hyperparameters["train_batch_size"],
        eval_batch_size=hyperparameters["eval_batch_size"]
        # No num_workers argument passed here
    )

    # Prepare with Accelerate
    # Note: No LR scheduler included in this simplified version
    model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, val_dataloader
    )

    # Output directory
    output_dir = hyperparameters["output_dir"]
    if accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
    accelerator.wait_for_everyone()

    # Training loop setup
    epochs_no_improve = 0
    min_val_loss = float('inf')

    accelerator.print(f"\n--- Starting Simplified Training ---")
    accelerator.print(f"!! MONITOR GPU VRAM & UTILIZATION (nvidia-smi) !!")
    accelerator.print(f"!! ADJUST train/eval batch sizes in hyperparameters for '{model_name}' if needed !!")

    # Assumes 'encoded_val_ds' is defined globally for validation loss trimming
    if 'encoded_val_ds' not in globals():
        print("Warning: 'encoded_val_ds' not found globally. Validation loss trimming might be inaccurate.")
        # Define a dummy length or handle absence if you want the code to run anyway
        # encoded_val_ds_len = float('inf') # Example fallback
    else:
        encoded_val_ds_len = len(encoded_val_ds)


    for epoch in range(hyperparameters["num_epochs"]):
        model.train()
        total_train_loss = 0
        progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_main_process)
        progress_bar.set_description(f"Epoch {epoch+1}/{hyperparameters['num_epochs']}")

        # --- Training Step ---
        for step, batch in enumerate(train_dataloader):
            outputs = model(**batch)
            loss = outputs.loss
            total_train_loss += loss.detach().float()

            accelerator.backward(loss) # Handles scaling if mixed precision is active
            optimizer.step()
            optimizer.zero_grad()

            progress_bar.set_postfix({'train_loss': loss.item()})
            progress_bar.update(1)

        avg_epoch_train_loss = total_train_loss / len(train_dataloader)
        accelerator.print(f"Epoch {epoch+1} Average Train Loss: {avg_epoch_train_loss:.4f}")

        # --- Evaluation Step ---
        model.eval()
        validation_losses = []
        eval_progress_bar = tqdm(total=len(val_dataloader), disable=not accelerator.is_main_process)
        eval_progress_bar.set_description("Evaluating")

        for batch in val_dataloader:
            with torch.no_grad():
                outputs = model(**batch)
            loss = outputs.loss
            # Gather using gather_for_metrics for robustness
            gathered_loss = accelerator.gather_for_metrics(loss.unsqueeze(0))
            validation_losses.append(gathered_loss)
            eval_progress_bar.update(1)

        val_loss = float('inf')
        if validation_losses:
            try:
                all_val_losses = torch.cat(validation_losses)
                # Attempt to trim padding potentially added by gather_for_metrics
                # Use the length obtained before the loop
                actual_val_size = encoded_val_ds_len
                if actual_val_size != float('inf'): # Only trim if length is valid
                    all_val_losses = all_val_losses[:actual_val_size]
                val_loss = all_val_losses.mean().item()
            except Exception as e:
                 accelerator.print(f"Note: Could not precisely trim validation losses ({e}). Using mean of all gathered.")
                 # Fallback if trimming fails
                 all_val_losses = torch.cat(validation_losses)
                 val_loss = all_val_losses.mean().item()


        accelerator.print(f"Epoch {epoch+1}: Validation Loss: {val_loss:.4f}")

        # --- Early Stopping & Saving ---
        if val_loss < min_val_loss:
            epochs_no_improve = 0
            min_val_loss = val_loss
            accelerator.print("Validation loss improved. Saving model...")
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            if accelerator.is_main_process:
                unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
                active_tokenizer.save_pretrained(output_dir)
            accelerator.print(f"Model and tokenizer saved to {output_dir}")
        else:
            epochs_no_improve += 1
            accelerator.print(f"Validation loss did not improve for {epochs_no_improve} epoch(s).")
            if epochs_no_improve >= hyperparameters["patience"]:
                accelerator.print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    accelerator.print("--- Simplified Training Finished ---")




--- Hyperparameters ---
model_checkpoint: google-t5/t5-base
learning_rate: 5e-05
num_epochs: 3
train_batch_size: 16
gradient_accumulation_steps: 1
eval_batch_size: 32
seed: 42
patience: 3
output_dir: ./content_simplifier_t5_base_simple/
mixed_precision: fp16
-----------------------


In [12]:
from accelerate import notebook_launcher
training_function()

[DEBUG] Value being passed to Accelerator for mixed_precision: fp16

--- Accelerator Info ---
Accelerator device: cuda
Number of processes: 1
Using mixed precision: fp16
------------------------
Using seed: 42
Loading model: google-t5/t5-base


loading configuration file config.json from cache at C:\Users\Ruben\.cache\huggingface\hub\models--google-t5--t5-base\snapshots\a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1\config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat


--- Starting Simplified Training ---
!! MONITOR GPU VRAM & UTILIZATION (nvidia-smi) !!
!! ADJUST train/eval batch sizes in hyperparameters for 'google-t5/t5-base' if needed !!


  0%|          | 0/45 [00:00<?, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


...

  0%|          | 0/3 [00:00<?, ?it/s]

Configuration saved in ./content_simplifier_t5_base_simple/config.json
Configuration saved in ./content_simplifier_t5_base_simple/generation_config.json


Epoch 3: Validation Loss: 0.7966
Validation loss improved. Saving model...


Model weights saved in ./content_simplifier_t5_base_simple/model.safetensors
tokenizer config file saved in ./content_simplifier_t5_base_simple/tokenizer_config.json
Special tokens file saved in ./content_simplifier_t5_base_simple/special_tokens_map.json
added tokens file saved in ./content_simplifier_t5_base_simple/added_tokens.json


Model and tokenizer saved to ./content_simplifier_t5_base_simple/
--- Simplified Training Finished ---


In [29]:
text = """Simplify: The patient presented with refractory ventricular tachycardia,
         unresponsive to amiodarone, necessitating synchronized cardioversion 
         and subsequent initiation of a continuous lidocaine infusion, 
         all within the context of underlying ischemic cardiomyopathy
         and severely reduced left ventricular ejection fraction.""" 

trained_model = T5ForConditionalGeneration.from_pretrained(r"C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\code\content_simplifier_t5_base_simple")

input_ids = tokenizer(text, return_tensors="pt").input_ids
 
generated_ids = trained_model.generate(input_ids, do_sample=True, 
    max_length=1000, 
    top_k=0, 
    temperature=0.7
)

summary = tokenizer.decode(generated_ids.squeeze(), skip_special_tokens=True)
print(summary)

loading configuration file C:\Users\Ruben\GPUcodig\LM\medical_project_simplification\code\content_simplifier_t5_base_simple\config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      

the patient presented with refractory ventricular tachycardia, unresponsive to amiodarone, necessitating synchronized cardioversion and subsequent initiation of a continuous lidocaine infusion, all within the context of underlying ischemic cardiomyopathy and severely reduced left ventricular ejection fraction .


In [30]:
print(summary)

the patient presented with refractory ventricular tachycardia, unresponsive to amiodarone, necessitating synchronized cardioversion and subsequent initiation of a continuous lidocaine infusion, all within the context of underlying ischemic cardiomyopathy and severely reduced left ventricular ejection fraction .
